# Online pose estimation — live camera

Same estimator, same calibration, frames arriving in real time. What changes is
everything around it: the camera has to be found and opened, the frame rate is set
by the sensor rather than by you, and a frame you drop is gone.

**This notebook is the live path.** It used to be a camera check that pointed at
`run_pose.py` for anything real; `run_pose.py` and `viz.py` are gone and their job is
done here, in one loop. The library modules are still libraries — this calls
`segment`, `estimator`, `filter`, `sources` and `recorder`, it does not reimplement them.

**The ELP global-shutter module (OV9281) changes the old picture.** Measured on this
machine: **121 fps at its native 1280×800**, **210 at 640×480** and **272 at 640×400**,
against the C270's ~28 fps flat. So the camera is no longer automatically the bottleneck,
and §2 says which one is binding. It is a **mono** sensor, which is not a limitation to work
around here but the thing the `dark` appearance is built for — see §1.

**Two rates, and they are not the same number.** `sources.CameraSource` runs a grabber
thread into a drop-oldest slot, so the sensor keeps delivering at its mode rate whatever the
estimator costs. Processing can never slow the camera; it only lowers how often a *pose*
comes out, and the frames it misses show up as `n_dropped`. Both are reported separately
below, because conflating them blames the camera for a slow estimator.

**Taking photos is a different notebook.** `camera/elp_capture.ipynb` has the preview and a capture button, for one camera or a stereo pair, and imports none of the estimator — so it works before any calibration exists, which is what the captures are needed for. This notebook is the pose loop; that one is the shutter.

## 0. Appearance, then imports

**Order matters and this is not cosmetic.** `estimator.RADIUS_MM` is bound at import time
from `segment.APPEARANCE`, so setting `POSE_APPEARANCE` *after* importing `estimator`
silently leaves you on the `bright` rig's radius — a wrong scale on every distance, with
nothing downstream able to detect it.

`dark` is the black-robot-on-white-backdrop rig with the coils in frame. It does not reduce
to a channel and a level: inverting the threshold finds the robot *and* the coils, the wires
and the room beyond the backdrop, all darker still, and a mono sensor has no chroma to tell
them apart. So it adds a **valid region** and a **spread limit** — both visible in §3's
overlay, which is the point of shading what was ignored.

In [ ]:
import os

# Before importing estimator. See the cell above.
os.environ["POSE_APPEARANCE"] = "dark"      # "bright" | "dark" | "red"

import sys, time, json, math
from collections import deque
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt

POSE = Path.cwd()
if POSE.name != "pose":                      # tolerate running from the repo root
    POSE = next(p for p in [POSE / "controller/pose", POSE / "ESP32_PMW/controller/pose"]
                if p.exists())
sys.path[:0] = [str(POSE), str(POSE / "validation")]

import conic, segment, estimator, zeroing, calibration, sources, recorder, bounds
from estimator import PoseEstimator, RADIUS_MM
from filter import PoseFilter
from recorder import PoseRecorder

RESULTS = POSE.parents[1] / "results" / "pose_validation"
K, dist = estimator.load_intrinsics()

bg = segment.load_background()
print(f"appearance   : {segment.APPEARANCE}")
print(f"rim radius   : {RADIUS_MM} mm   (bound at import from the appearance)")
print(f"axial weight : {segment.AXIAL_DEFAULT}")
if segment.APPEARANCE == "dark":
    print(f"dark level   : {segment.DARK_THRESH}  (robot must read below "
          f"{255 - segment.DARK_THRESH} counts; passes over 170-215)")
    print(f"region       : {'background subtraction' if bg is not None else 'backdrop finder (no background frame)'}"
          f"   spread {segment.DARK_MAX_SPREAD}")

## 1. Find and open the camera

macOS enumerates USB cameras *before* the built-in FaceTime, so the ELP is `"camera:0"` and
the laptop camera is `"camera:1"`. The first open triggers the permission prompt; without it
you get black frames, which the preview will show.

**Ask for 1280×800, not 720p.** 1280×720 is a windowed readout of the native 800, so it is
slower *and* narrower. Only 640×400 is a true 0.5× rescale, so `rig.Camera.scaled(0.5)`
carries intrinsics from 1280×800 to 640×400 and nowhere else; the crop modes keep fx, fy and
shift cx, cy instead.

**640×400 is the better operating point if update rate matters more than resolution.**
Measured single-core, `segment()` costs 7.9 ms at 1280×800 against an 8.3 ms camera period —
it only just fits — while at 640×400 it costs 2.5 ms against 3.7 ms and the loop stays
camera-bound. Change `WIDTH`/`HEIGHT` here and nothing else needs to move.

`grayscale=True` is left at its default and is correct for this appearance: the sensor is
mono, so there is no colour being discarded. (The `red` appearance is the one that needs
`grayscale=False`; asking for colour here would only cost a conversion.)

In [ ]:
CAM = "camera:0"
WIDTH, HEIGHT = 1280, 800

cam = sources.open_source(CAM, width=WIDTH, height=HEIGHT)
print("asked", f"{WIDTH}x{HEIGHT}", " got", cam.actual)

## 2. What rate is the camera actually giving you?

Measured over real reads, not the driver's advertised number. This is the ceiling
everything downstream lives under — and it is the *camera's* number, before the estimator
has been asked for anything.

In [ ]:
fps, n_read = sources.measure_fps(cam, n=120)

if not fps:
    print("no frames -- the camera is not delivering")
else:
    seg_budget = 1e3 / fps
    print(f"measured {fps:.1f} fps over {n_read} reads -> {seg_budget:.2f} ms per frame")
    print(f"the estimator needs ~1-8 ms depending on resolution, so at this rate the")
    print(f"binding constraint is {'the camera' if seg_budget > 8 else 'COMPUTE, not the camera'}")

## 3. The live loop

One loop, doing everything: read, estimate, filter, draw, and optionally record. There used
to be a preview loop and a separate capture loop, which is two places for the two to drift
apart.

What the overlay shows, and why each part is there:

- **the fitted ellipse and its axes** — `segment.draw`, unchanged;
- **the rotor axis** as a cyan arrow, by projecting `xyz_mm` and `xyz_mm + L·normal` through
  `K`. This is the one thing `segment.draw` cannot do for itself: it needs the camera matrix
  and the 3-D pose, neither of which segmentation knows about;
- **the ignored area shaded red** — everything outside the valid region. A wrong pose and a
  wrong *rejection* look identical in a plain ellipse overlay, and they have opposite fixes:
  a bad pose means retune the fit, a bad region means fix the lighting or the framing;
- **sustained fps over a rolling window**, not a cumulative mean. A cumulative average hides
  a mid-run stall, which is exactly the failure this display exists to reveal. Camera rate
  and pose rate are shown separately.

The JPEG encode is throttled to `DISPLAY_HZ`. Encoding every frame at 121 fps costs more than
the estimator does, and would make the number on screen a measure of the display rather than
of the pipeline. Bounded by frame count *and* wall clock, so a stalled camera cannot hang the
kernel. The estimator keeps its branch history across the run — that is what resolves the
ambiguity — so do not construct it inside the loop.

In [ ]:
from IPython.display import Image, display

N_FRAMES, MAX_SECONDS = 2000, 30.0
DISPLAY_HZ = 15.0            # JPEG encode is not free; see the cell above
FPS_WINDOW = 60              # frames in the rolling rate estimate
RECORD = False               # True also writes live_poses.csv
NORMAL_MM = 25.0             # length of the drawn rotor axis, in mm


def normal_segment_px(pose, K, length_mm=NORMAL_MM):
    """The rotor axis as ((x0,y0),(x1,y1)) in pixels, or None behind the camera."""
    p0 = np.asarray(pose.xyz_mm, dtype=float)
    p1 = p0 + length_mm * np.asarray(pose.normal, dtype=float)
    if p0[2] <= 1e-6 or p1[2] <= 1e-6:
        return None
    uv = (K @ np.stack([p0, p1]).T).T
    uv = uv[:, :2] / uv[:, 2:3]
    return tuple(map(tuple, uv))


est = PoseEstimator(camera_matrix=K, dist_coeffs=dist)
filt = PoseFilter()
rows, t_ms, lost = [], [], 0
intervals = deque(maxlen=FPS_WINDOW)
handle = None
t_start = time.monotonic()
t_prev = t_start
t_drawn = 0.0
n_grabbed0 = getattr(cam, "n_grabbed", 0)

rec = (PoseRecorder(str(POSE / "live_poses.csv"),
                    meta={"source": CAM, "appearance": segment.APPEARANCE,
                          "radius_mm": RADIUS_MM, "axial": segment.AXIAL_DEFAULT,
                          "resolution": f"{WIDTH}x{HEIGHT}", "fps_measured": fps})
       if RECORD else None)

try:
    for i in range(N_FRAMES):
        now = time.monotonic()
        if now - t_start > MAX_SECONDS:
            print("stopped on the time limit")
            break
        item = cam.read()
        if item is None:
            print("source ended")
            break
        t_cap, frame = item
        intervals.append(now - t_prev)
        t_prev = now

        pose = est.update(frame, t=t_cap)
        if pose is None:
            lost += 1
        else:
            filt.update(pose, t=t_cap)
            t_ms.append(pose.t_seg_ms + pose.t_est_ms)
            rows.append([t_cap, *pose.xyz_mm, pose.theta_deg, pose.phi_deg, pose.psi_deg])
            if rec is not None:
                rec.write(pose)

        if now - t_drawn >= 1.0 / DISPLAY_HZ:
            t_drawn = now
            loop_hz = len(intervals) / max(sum(intervals), 1e-9)
            grabbed = getattr(cam, "n_grabbed", 0) - n_grabbed0
            cam_hz = grabbed / max(now - t_start, 1e-9)
            seg = pose.extra.get("segmentation") if pose is not None else None
            out = segment.draw(frame, seg,
                               normal_px=normal_segment_px(pose, K) if pose is not None else None)
            head = (f"loop {loop_hz:5.1f} Hz   camera {cam_hz:5.1f} Hz   "
                    f"dropped {getattr(cam, 'n_dropped', 0)}   lost {lost}")
            cv2.putText(out, head, (12, out.shape[0] - 34),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 1)
            if pose is not None:
                x, y, z = pose.xyz_mm
                cv2.putText(out, f"xyz {x:+7.1f} {y:+7.1f} {z:+7.1f} mm   "
                                 f"tilt {pose.theta_deg:5.1f}  az {pose.phi_deg:+6.1f}",
                            (12, out.shape[0] - 14), cv2.FONT_HERSHEY_SIMPLEX,
                            0.55, (0, 255, 255), 1)
            img = Image(data=cv2.imencode(".jpg", out)[1].tobytes())
            handle = display(img, display_id=True) if handle is None else (handle.update(img) or handle)
finally:
    if rec is not None:
        rec.close()

wall = time.monotonic() - t_start
print(f"\n{len(rows)} poses, {lost} lost, over {wall:.1f} s")
if t_ms:
    print(f"pipeline {np.mean(t_ms):.2f} ms/frame (median {np.median(t_ms):.2f}) "
          f"-> {1e3 / np.mean(t_ms):.0f} Hz if nothing else were in the way")
print(f"camera grabbed {getattr(cam, 'n_grabbed', 0) - n_grabbed0}, "
      f"consumer missed {getattr(cam, 'n_dropped', 0)}")

## 4. What is being ignored

The overlay shades the rejected area while the loop runs, which is the right place to catch
it. This is the same thing held still: the frame, the valid region, and the thresholded
silhouette that the ellipse is fitted to.

If the region has swallowed a coil, or collapsed onto a fraction of the backdrop, it shows
here immediately — and neither is visible in a pose trace, which will look perfectly smooth
while being wrong.

In [ ]:
item = cam.read()
frame = item[1] if item is not None else None

if frame is None:
    print("no frame")
elif segment.APPEARANCE != "dark":
    print(f"appearance is '{segment.APPEARANCE}', which looks at the whole frame -- nothing to show")
else:
    g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame
    region = segment.valid_region(g)
    seg = segment.segment(g)
    ch, lvl = segment.score_channel(g, region=region)
    sil = cv2.threshold(ch, lvl, 255, cv2.THRESH_BINARY)[1] if ch is not None else np.zeros_like(g)

    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    ax[0].imshow(segment.draw(g, seg)[:, :, ::-1]); ax[0].set_title("frame + fit + ignored")
    ax[1].imshow(region if region is not None else np.zeros_like(g), cmap="gray")
    ax[1].set_title(f"valid region ({0.0 if region is None else 100 * region.mean() / 255:.0f}% of frame)")
    ax[2].imshow(sil, cmap="gray"); ax[2].set_title(f"silhouette (level {lvl})")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()

    if seg is None:
        print("no detection -- if the region looks wrong, fix the lighting before the constants")
    else:
        print(f"major {seg.ellipse[1][0]:.1f} px  minor {seg.ellipse[1][1]:.1f} px  "
              f"rms {seg.fit_rms_px:.2f} px  from {seg.n_points} hull points  ({seg.t_ms:.2f} ms)")

In [ ]:
cam.close()
print("camera released")

## 5. Look at the run

Same six channels as the offline notebook. A live capture of a *stationary* robot
is the most useful diagnostic here: every wiggle in these traces is noise plus
silhouette variation, with no real motion underneath, so the spread is a direct
read of the estimator's repeatability on your actual hardware.

In [ ]:
a = np.array(rows, dtype=float)
t = a[:, 0] - a[0, 0]
names = ["x (mm)", "y (mm)", "z (mm)", "tilt θ (deg)", "azimuth φ (deg)", "ψ (deg)"]

fig, axes = plt.subplots(3, 2, figsize=(11, 7), sharex=True)
for k, (ax, nm) in enumerate(zip(axes.T.ravel(), names)):
    ax.plot(t, a[:, 1 + k], lw=0.9)
    ax.set_ylabel(nm); ax.grid(alpha=0.3)
for ax in axes[-1]:
    ax.set_xlabel("time (s)")
plt.tight_layout(); plt.show()

print("standard deviation over the run (stationary robot => this is repeatability):")
for k, nm in enumerate(names):
    print(f"  {nm:<18} {np.std(a[:, 1 + k]):.3f}")

## 6. Repeatability against the floor

If the robot was stationary, the spread above is measurement noise, and `bounds.py`
says what it could be. Expect the measured value to sit well above the floor: on
rendered data it is ~23× the photon bound, because the limit is that the silhouette
is not the rim, not that the edge is poorly located.

A measured spread far *worse* than that, though, points at something local —
focus, exposure, a partly occluded rim — and is worth chasing.

In [ ]:
if len(rows) > 5:
    z_med = float(np.median(a[:, 3]))
    b = bounds.budget(z_med, RADIUS_MM, K, tilt_deg=float(np.median(a[:, 4])))
    lat = float(np.hypot(np.std(a[:, 1]), np.std(a[:, 2])))
    print(f"range {z_med:.0f} mm, rim {2 * b['semi_major_px']:.0f} px")
    print(f"  lateral spread measured {lat:.3f} mm")
    print(f"  depth   spread measured {np.std(a[:, 3]):.3f} mm")
    print(f"  depth/lateral measured  {np.std(a[:, 3]) / max(lat, 1e-9):.1f}x"
          f"   predicted {b['depth_lateral_ratio']:.1f}x")
    print("\n(the ratio is the geometric prediction and does not depend on how")
    print(" noisy your setup is -- if it is far off, suspect the calibration)")

## 7. Longer runs, and stereo

Set `RECORD = True` in §3 and raise `MAX_SECONDS`. The CSV lands at
`controller/pose/live_poses.csv` with a provenance header (`recorder.write_metadata`), and
reads back with `pandas.read_csv(path, comment="#")`; lost frames are written as blank rows
rather than dropped, so the time base stays honest.

Stereo is the same loop with two changes: `sources.open_stereo` in place of `open_source`,
and `stereo.StereoPoseEstimator` in place of `PoseEstimator`. The cell below is the whole
difference, and it is **guarded** — with one camera mounted it says so and stops, rather than
failing somewhere less obvious.

The number to watch is `skew_stats()`. Two free-running USB cameras are not synchronised;
`StereoSource` measures how far apart each pair actually landed instead of assuming. At
121 fps one frame of skew is 8.3 ms, and the robot moves 15–22 mm/s in hover, so that is
0.15 mm — inside the budget, but only because the motion is slow. It is not inside it during
a climb, which is what `filter.PoseFilter.predict_ahead` exists for.

`controller/elp/capture.py` records the paired frames that
`controller/pose/calibrate_stereo.py` calibrates into `stereo_rig.json`.

In [ ]:
import rig as rigmod
import stereo as stereomod

STEREO_SOURCES = ["camera:0", "camera:1"]
STEREO_SECONDS = 10.0

rig_path = POSE / "stereo_rig.json"
if not rig_path.exists():
    print(f"no {rig_path.name} -- calibrate first:")
    print("  uv run python controller/elp/capture.py --index 0 1 --board --count 25")
    print("  uv run python controller/elp/calibrate.py --session <that>")
    print("  uv run python controller/pose/calibrate_stereo.py --cam-a ... --cam-b ...")
else:
    pair = sources.open_stereo(STEREO_SOURCES, width=WIDTH, height=HEIGHT)
    sest = stereomod.StereoPoseEstimator(rigmod.StereoRig.load(rig_path))
    n, lost_s = 0, 0
    t0 = time.monotonic()
    try:
        while time.monotonic() - t0 < STEREO_SECONDS:
            item = pair.read()
            if item is None:
                break
            t_cap, frames = item
            p = sest.update(frames, t=t_cap)
            n += 1
            if p is None:
                lost_s += 1
    finally:
        pair.close()
    print(f"{n} pairs, {lost_s} lost, over {time.monotonic() - t0:.1f} s")
    print("capture skew:", pair.skew_stats())